# Detección de tuberculosis con características radiómicas

Entrenamiento y evaluación de los clasificadores de TB a partir de las características radiómicas extraídas dentro de cada segmentación pulmonar.

La pregunta del proyecto no es qué clasificador es mejor, sino **si la técnica de segmentación cambia el desempeño de la detección de TB**. Por eso, cada uno de los clasificadores es entrenado una vez basado en cada una de las técnicas de segmentación utilizando las mismas características (mismo protocolo, folds y busqueda de hiperparámetros). Entre pruebas, solamente cambia la técnica de segmentación de la que vienen las características.


| Sección | Contenido |
|---|---|
| 1 | Configuración del experimento |
| 2 | Entrada: archivos de características |
| 3 | Etiquetas y folds |
| 4 | Carga de características |
| 5 | Clasificadores |
| 6 | Validación cruzada anidada |
| 7 | Resultados |
| 8 | Comparación entre técnicas de segmentación |
| 9 | Importancia de características |
| 10 | Guardado |
| 11 | Qué hay que tener en cuenta |

## 1. Configuración

Todo lo que cambia entre experimentos está en la celda siguiente:

- `DATASET`: `"montgomery_shenzhen"` (704 imágenes) o `"tbx11k"` (1600 imágenes de `subset.csv`).
- `METHODS`: técnicas de segmentación a comparar. Cada una necesita un archivo de características por familia (sección 2).
- `FEATURE_CLASSES`: familias de pyradiomics que entran a los modelos. Por defecto solo **primer orden** (`firstorder`): estadísticas del histograma de intensidades dentro de la máscara.
- `IMAGE_TYPES`: imagen de la que salen las características. `original` es la radiografía sin filtrar; los filtros de pyradiomics (`wavelet`, `log-sigma`, …) se agregan aquí si se extraen.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from tbseg.config import MS_INDEX_CSV, N_FOLDS, RESULTS_DIR, ROOT, SEED, SUBSET_CSV

# ------------------------------------------------------------------ experimento
DATASET = "montgomery_shenzhen"                  # "montgomery_shenzhen" o "tbx11k"
METHODS = ["unet", "attention_unet", "medsam"]   # técnicas de segmentación; la primera es la referencia (sección 8)
FEATURE_CLASSES = ["firstorder"]                 # firstorder, glcm, glrlm, glszm, gldm, ngtdm, shape2D
IMAGE_TYPES = ["original"]                       # original, wavelet-LLH, log-sigma-1-0-mm-3D, …

INNER_FOLDS = 3                                  # folds internos para elegir hiperparámetros (sección 6)

assert DATASET in ("montgomery_shenzhen", "tbx11k"), DATASET
RADIOMICS_DIR = RESULTS_DIR / "radiomics" / DATASET
RADIOMICS_PREFIX = {"montgomery_shenzhen": "Montgomery_shenzhen", "tbx11k": "TBX11K"}[DATASET]   # inicio del nombre de cada CSV (sección 2)
CLF_RESULTS_DIR = RESULTS_DIR / "classification" / DATASET / f"{'+'.join(IMAGE_TYPES)}_{'+'.join(FEATURE_CLASSES)}"

display(pd.Series({
    "Dataset": DATASET,
    "Técnicas de segmentación": ", ".join(METHODS),
    "Tipos de imagen": ", ".join(IMAGE_TYPES),
    "Familias de características": ", ".join(FEATURE_CLASSES),
    "Folds externos / internos": f"{N_FOLDS} / {INNER_FOLDS}",
    "Semilla": SEED,
    "Características": str(RADIOMICS_DIR.relative_to(ROOT)),
}).to_frame("valor").rename_axis("parámetro"))

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
METHOD_NAMES = {"unet": "U-Net", "attention_unet": "Attention U-Net", "medsam": "MedSAM"}
METHOD_COLORS = {"unet": "#2a78d6", "attention_unet": "#1baf7a", "medsam": "#eb6834"}
MODEL_NAMES = {"svm": "SVM", "random_forest": "Random Forest"}
NAMES = {**METHOD_NAMES, **MODEL_NAMES}


def name(key):
    return NAMES.get(key, key)

## 2. Entrada: archivos de características

Contrato con el notebook de extracción radiómica (pendiente). Un archivo por técnica de segmentación y familia de características:

```
results/radiomics/<dataset>/<prefijo>_<familia>_<técnica>.csv
p. ej. results/radiomics/montgomery_shenzhen/Montgomery_shenzhen_firstorder_unet.csv
```

| `DATASET` | `<prefijo>` |
|---|---|
| `montgomery_shenzhen` | `Montgomery_shenzhen` |
| `tbx11k` | `TBX11K` |

El nombre distingue mayúsculas de minúsculas.

| Columna | Contenido |
|---|---|
| `image_id` | Identificador de la imagen, igual que en el índice del dataset. Una fila por imagen. |
| `<tipo de imagen>_<familia>_<característica>` | Nombre que entrega pyradiomics, p. ej. `original_firstorder_Mean`. |
| `diagnostics_*` | Opcionales. Se ignoran. |

Reglas:
- **La etiqueta y los folds no vienen en este archivo.** Se toman del índice del dataset (sección 3), una sola fuente para todas las técnicas.
- **Todas las técnicas deben cubrir las mismas imágenes.** Si a una técnica le falta una imagen (p. ej. máscara vacía y pyradiomics falla), el notebook se detiene: comparar técnicas evaluadas sobre imágenes distintas mezcla el efecto de la técnica con el de la muestra.
- **Mismas opciones de pyradiomics para todas las técnicas** (discretización, normalización, `force2D`). Solo cambia la máscara.

Con la configuración por defecto de pyradiomics, la familia `firstorder` entrega 18 características con el prefijo `original_firstorder_`, y las columnas de diagnóstico empiezan por `diagnostics_`.

## 3. Etiquetas y folds

Cada dataset se reduce a la misma tabla: una fila por imagen con `image_id`, `label` (1 = TB), `group` (paciente), `stratum` (subgrupo para reportar) y `fold`.

| | Montgomery-Shenzhen | TBX11K |
|---|---|---|
| Índice | `data/metadata/montgomery_shenzhen_index.csv` | `data/metadata/subset.csv` |
| `label` | `tb` | `label` (TB frente a sanas + enfermas no-TB) |
| `group` | `patient_id` | `image_id` (el subconjunto no trae identificador de paciente) |
| `stratum` | `source` (montgomery / shenzhen) | `source_class` (tb / health / sick) |
| `fold` | Se calcula aquí | Ya fijado en `subset.csv` (`python -m tbseg.data`) |

En Montgomery-Shenzhen los folds se calculan con `StratifiedGroupKFold`:
- **Agrupado por paciente**: las radiografías de un mismo paciente quedan en el mismo fold, para no evaluar con un paciente que el modelo ya vio.
- **Estratificado por fuente y diagnóstico a la vez**: cada fold conserva la proporción de Montgomery/Shenzhen y de TB/normal. Las fuentes difieren en adquisición (notebook de exploración, sección 8), así que un fold con una mezcla distinta de fuentes mediría otra cosa.

Los folds son **los mismos para todas las técnicas y todos los clasificadores**. Esto permite comparar técnicas fold a fold (sección 8).

In [ ]:
def load_metadata(dataset: str) -> pd.DataFrame:
    # Una fila por imagen: image_id, label (1 = TB), group (paciente), stratum (subgrupo) y fold
    cols = ["image_id", "label", "group", "stratum", "fold"]
    if dataset == "tbx11k":
        meta = pd.read_csv(SUBSET_CSV).rename(columns={"source_class": "stratum"})
        meta["group"] = meta["image_id"]
        return meta[cols]

    meta = pd.read_csv(MS_INDEX_CSV).rename(columns={"tb": "label", "patient_id": "group", "source": "stratum"})
    strata = meta["stratum"] + "_" + meta["label"].astype(str)
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    meta["fold"] = -1
    for fold, (_, test_idx) in enumerate(cv.split(meta, strata, groups=meta["group"])):
        meta.loc[test_idx, "fold"] = fold
    return meta[cols]


meta = load_metadata(DATASET)
assert (meta.groupby("group")["fold"].nunique() == 1).all(), "un paciente quedó en dos folds"
patients = "sin identificador de paciente" if DATASET == "tbx11k" else f"{meta['group'].nunique()} pacientes"
print(f"{DATASET}: {len(meta)} imágenes ({patients}) | TB {meta['label'].sum()} | no TB {(meta['label'] == 0).sum()}")
display(pd.crosstab([meta["stratum"], meta["label"].map({0: "no TB", 1: "TB"})], meta["fold"],
                    margins=True, margins_name="total"))

## 4. Carga de características

`load_features` lee, para una técnica, un archivo por cada familia de `FEATURE_CLASSES`, comprueba el contrato de la sección 2 y une las columnas en una matriz alineada fila a fila con `meta`. De cada archivo solo conserva las columnas de su familia cuyo tipo de imagen esté en `IMAGE_TYPES`.

In [ ]:
def feature_columns(columns, feature_class: str) -> list:
    # Columnas <tipo de imagen>_<familia>_<característica> de una familia
    selected = []
    for col in columns:
        parts = col.split("_", 2)
        if len(parts) == 3 and parts[0] in IMAGE_TYPES and parts[1] == feature_class:
            selected.append(col)
    return selected


def load_feature_class(method: str, feature_class: str, meta: pd.DataFrame) -> pd.DataFrame:
    # Características de una técnica y una familia, en el mismo orden de filas que meta
    path = RADIOMICS_DIR / f"{RADIOMICS_PREFIX}_{feature_class}_{method}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No existe {path}. Falta la extracción radiómica de '{method}'.")
    feats = pd.read_csv(path)

    if feats["image_id"].duplicated().any():
        raise ValueError(f"{path.name}: hay image_id repetidos")
    missing = sorted(set(meta["image_id"]) - set(feats["image_id"]))
    if missing:
        raise ValueError(f"{path.name}: faltan {len(missing)} imágenes del índice, p. ej. {missing[:5]}")
    cols = feature_columns(feats.columns, feature_class)
    if not cols:
        raise ValueError(f"{path.name}: ninguna columna coincide con IMAGE_TYPES={IMAGE_TYPES} "
                         f"y la familia '{feature_class}'")
    return meta[["image_id"]].merge(feats, on="image_id", how="left")[cols].astype(float)


def load_features(method: str, meta: pd.DataFrame) -> pd.DataFrame:
    # Matriz de características de una técnica: une el archivo de cada familia elegida
    return pd.concat([load_feature_class(method, cls, meta) for cls in FEATURE_CLASSES], axis=1)


features = {method: load_features(method, meta) for method in METHODS}
display(pd.DataFrame({
    "características": {m: X.shape[1] for m, X in features.items()},
    "valores faltantes": {m: int(X.isna().sum().sum()) for m, X in features.items()},
    "constantes": {m: int((X.nunique() <= 1).sum()) for m, X in features.items()},
}).rename(index=name))

## 5. Clasificadores

Cada clasificador es un `Pipeline` de scikit-learn. Todo el preprocesamiento va dentro del pipeline, así que se ajusta **solo con las imágenes de entrenamiento de cada fold**; las de prueba no influyen en la imputación, en la eliminación de constantes ni en la estandarización.

| Paso | SVM | Random Forest | Para qué |
|---|---|---|---|
| `SimpleImputer(median)` | ✓ | ✓ | Rellena valores faltantes si pyradiomics no pudo calcular alguna característica. |
| `VarianceThreshold()` | ✓ | ✓ | Elimina características constantes en el fold de entrenamiento. |
| `StandardScaler()` | ✓ | | El kernel RBF usa distancias: sin escalar, la característica de mayor rango domina. Los árboles no dependen de la escala. |
| Clasificador | `SVC(kernel="rbf")` | `RandomForestClassifier(300 árboles)` | |

`class_weight="balanced"` pondera las clases según su frecuencia en el fold de entrenamiento. Con los datasets actuales cambia poco (345 TB / 359 normales en Montgomery-Shenzhen, 800 / 800 en TBX11K), pero evita que un desbalance futuro sesgue el modelo hacia la clase mayoritaria.

Los hiperparámetros se eligen con la rejilla de cada modelo (sección 6). **Para agregar un clasificador** (p. ej. una red neuronal) basta con agregar una entrada a `MODELS`: el resto del notebook la recorre automáticamente.

In [ ]:
MODELS = {
    "svm": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("variance", VarianceThreshold()),
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", class_weight="balanced")),
        ]),
        {"clf__C": [0.1, 1, 10, 100], "clf__gamma": ["scale", 0.001, 0.01, 0.1]},
    ),
    "random_forest": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("variance", VarianceThreshold()),
            ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED)),
        ]),
        {"clf__max_depth": [None, 10], "clf__min_samples_leaf": [1, 5], "clf__max_features": ["sqrt", 0.5]},
    ),
}

display(pd.DataFrame([
    {"modelo": name(m), "rejilla": "; ".join(f"{k.removeprefix('clf__')}: {v}" for k, v in grid.items()),
     "combinaciones": int(np.prod([len(v) for v in grid.values()]))}
    for m, (_, grid) in MODELS.items()
]).set_index("modelo"))

## 6. Validación cruzada anidada

Si los hiperparámetros se eligen mirando el fold de prueba, el desempeño reportado es optimista. La validación cruzada **anidada** separa las dos decisiones:

```
para cada fold externo k (5 en total):
    prueba        = imágenes del fold k
    entrenamiento = los otros 4 folds
        └─ GridSearchCV con 3 folds internos (agrupados por paciente) elige los hiperparámetros por AUC
        └─ reentrena con todo el entrenamiento y predice el fold k una sola vez
```

El fold externo no participa en ninguna decisión. Cada imagen recibe **una predicción**, hecha por un modelo que no la vio.

**Métricas** (por fold externo):
- **AUC** (métrica principal): probabilidad de que una radiografía TB reciba un puntaje mayor que una no-TB. No depende del umbral.
- **Sensibilidad**: fracción de casos TB detectados. En tamizaje es la que más importa: un caso no detectado no recibe tratamiento.
- **Especificidad**: fracción de casos no-TB descartados correctamente.
- **Exactitud balanceada**: media de sensibilidad y especificidad.
- **F1**: media armónica de precisión y sensibilidad.

Sensibilidad, especificidad, exactitud balanceada y F1 usan el umbral por defecto de cada modelo (SVM: distancia 0 al hiperplano; Random Forest: probabilidad 0,5).

In [ ]:
def positive_score(model, X):
    # Puntaje de la clase TB: probabilidad (Random Forest) o distancia al hiperplano (SVM)
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


def fold_metrics(y, pred, score) -> dict:
    return {
        "auc": roc_auc_score(y, score),
        "sensibilidad": recall_score(y, pred, pos_label=1),
        "especificidad": recall_score(y, pred, pos_label=0),
        "exactitud_balanceada": balanced_accuracy_score(y, pred),
        "f1": f1_score(y, pred),
    }


def run_cv(method: str, model_name: str):
    # Validación cruzada anidada de un clasificador con las características de una técnica
    X, y, groups = features[method], meta["label"].to_numpy(), meta["group"].to_numpy()
    pipeline, grid = MODELS[model_name]
    run = {"method": method, "model": model_name}
    predictions, metrics, params, importances = [], [], [], []

    for fold in range(N_FOLDS):
        train, test = (meta["fold"] != fold).to_numpy(), (meta["fold"] == fold).to_numpy()
        inner_cv = StratifiedGroupKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=SEED)
        search = GridSearchCV(clone(pipeline), grid, scoring="roc_auc", cv=inner_cv, n_jobs=-1)
        search.fit(X[train], y[train], groups=groups[train])

        score, pred = positive_score(search, X[test]), search.predict(X[test])
        predictions.append(meta.loc[test, ["image_id", "stratum", "label"]].assign(**run, fold=fold, score=score, pred=pred))
        metrics.append({**run, "fold": fold, "n_test": int(test.sum()), "auc_interna": search.best_score_,
                        **fold_metrics(y[test], pred, score)})
        params.append({**run, "fold": fold, "params": str({k.removeprefix("clf__"): v for k, v in search.best_params_.items()})})

        best = search.best_estimator_
        if hasattr(best[-1], "feature_importances_"):
            importances.append(pd.DataFrame({**run, "fold": fold, "feature": best[:-1].get_feature_names_out(),
                                             "importance": best[-1].feature_importances_}))

    return (pd.concat(predictions), pd.DataFrame(metrics), pd.DataFrame(params),
            pd.concat(importances) if importances else pd.DataFrame())

In [ ]:
results = {"predictions": [], "metrics": [], "params": [], "importances": []}
start = time.time()
for method in METHODS:
    for model_name in MODELS:
        t0 = time.time()
        for key, df in zip(results, run_cv(method, model_name)):
            results[key].append(df)
        auc = results["metrics"][-1]["auc"]
        print(f"{name(method):>16} | {name(model_name):<13} | AUC {auc.mean():.3f} ± {auc.std():.3f} | {time.time() - t0:.0f} s")
print(f"Total: {(time.time() - start) / 60:.1f} min")


def run_order(df):
    # Ordena por modelo y luego por técnica, en el orden de MODELS y METHODS
    rank = {key: i for i, key in enumerate([*MODELS, *METHODS])}
    return df.sort_values(["model", "method"], key=lambda s: s.map(rank), kind="stable", ignore_index=True)


predictions, metrics, params, importances = (
    run_order(df) if len(df) else df for df in (pd.concat(v, ignore_index=True) for v in results.values()))

## 7. Resultados

### 7.1 Resumen

Media ± desviación entre los folds externos. La desviación mide cuánto cambia el desempeño según qué imágenes caen en prueba; es la escala con la que hay que leer cualquier diferencia entre técnicas.

In [ ]:
METRICS = ["auc", "sensibilidad", "especificidad", "exactitud_balanceada", "f1"]
agg = metrics.groupby(["model", "method"], sort=False)[METRICS].agg(["mean", "std"])
summary = pd.DataFrame({m: agg[(m, "mean")].map("{:.3f}".format) + " ± " + agg[(m, "std")].map("{:.3f}".format)
                        for m in METRICS})
display(summary.rename(index=name))

### 7.2 AUC por fold

Cada línea gris une el mismo fold externo a través de las técnicas: las imágenes de prueba son idénticas, así que la pendiente de la línea es el efecto de la técnica de segmentación en ese fold.

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5.5 * len(MODELS), 3.8), sharey=True, squeeze=False)
for ax, model_name in zip(axes[0], MODELS):
    auc = metrics[metrics["model"] == model_name].pivot(index="fold", columns="method", values="auc")[METHODS]
    x = np.arange(len(METHODS))
    for _, row in auc.iterrows():
        ax.plot(x, row.to_numpy(), color="0.75", lw=1, zorder=1)
    for i, method in enumerate(METHODS):
        ax.scatter(np.full(len(auc), i), auc[method], color=METHOD_COLORS.get(method), s=30, zorder=2)
        ax.hlines(auc[method].mean(), i - 0.25, i + 0.25, color=METHOD_COLORS.get(method), lw=2.5, zorder=3)
    ax.set_xticks(x, [name(m) for m in METHODS])
    ax.set_title(name(model_name), loc="left"); ax.grid(axis="y", alpha=0.3)
axes[0, 0].set_ylabel("AUC (fold externo)")
fig.suptitle("AUC por fold; barra: media", x=0.01, ha="left")
fig.tight_layout()
plt.show()

### 7.3 Curvas ROC

Curva media entre folds (banda: ± una desviación). Cada fold tiene su propia curva y luego se promedian en una rejilla común de falsos positivos. No se juntan los puntajes de todos los folds en una sola curva porque la escala del puntaje del SVM cambia con los hiperparámetros elegidos en cada fold.

In [ ]:
FPR_GRID = np.linspace(0, 1, 101)


def mean_roc(df):
    tprs = []
    for _, f in df.groupby("fold"):
        fpr, tpr, _ = roc_curve(f["label"], f["score"])
        tprs.append(np.interp(FPR_GRID, fpr, tpr))
    tprs = np.array(tprs)
    tprs[:, 0] = 0
    return tprs.mean(axis=0), tprs.std(axis=0)


fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.4), squeeze=False)
for ax, model_name in zip(axes[0], MODELS):
    for method in METHODS:
        sel = (predictions["model"] == model_name) & (predictions["method"] == method)
        tpr, sd = mean_roc(predictions[sel])
        auc = metrics.loc[(metrics["model"] == model_name) & (metrics["method"] == method), "auc"].mean()
        line, = ax.plot(FPR_GRID, tpr, lw=2, color=METHOD_COLORS.get(method), label=f"{name(method)} (AUC {auc:.3f})")
        ax.fill_between(FPR_GRID, tpr - sd, np.minimum(tpr + sd, 1), color=line.get_color(), alpha=0.12)
    ax.plot([0, 1], [0, 1], ls="--", color="0.6", lw=1)
    ax.set_xlabel("1 − especificidad"); ax.set_ylabel("Sensibilidad")
    ax.set_title(name(model_name), loc="left"); ax.legend(frameon=False, loc="lower right", fontsize=8)
fig.suptitle("Curva ROC media entre folds", x=0.01, ha="left")
fig.tight_layout()
plt.show()

### 7.4 Desempeño por subgrupo

El promedio puede esconder que un modelo funciona en un subgrupo y falla en otro.
- **Montgomery-Shenzhen**: por fuente. Las fuentes difieren en adquisición; si el AUC cae en una de ellas, el modelo puede estar aprendiendo la fuente y no la enfermedad.
- **TBX11K**: por clase de origen. Cada subgrupo tiene una sola clase, así que la AUC no está definida y se reporta el **acierto**: en `tb` equivale a la sensibilidad; en `health` y `sick`, a la especificidad. La diferencia entre `health` y `sick` indica si los pacientes con otra enfermedad pulmonar se confunden con TB.

Se calcula por fold y se promedia entre folds.

In [ ]:
def stratum_metrics(df):
    out = {"imágenes": len(df), "acierto": (df["label"] == df["pred"]).mean()}
    out["auc"] = roc_auc_score(df["label"], df["score"]) if df["label"].nunique() == 2 else np.nan
    return pd.Series(out)


by_stratum = (predictions.groupby(["model", "method", "stratum", "fold"], sort=False)
              .apply(stratum_metrics, include_groups=False)
              .groupby(["model", "method", "stratum"], sort=False)
              .agg(imágenes=("imágenes", "sum"), acierto=("acierto", "mean"), auc=("auc", "mean"))
              .astype({"imágenes": int}))
display(by_stratum.unstack("stratum").dropna(axis=1, how="all").round(3).rename(index=name))

### 7.5 Hiperparámetros elegidos

Cuántas veces (de 5 folds externos) se eligió cada combinación. Si cambia en cada fold, el desempeño es poco sensible a ese hiperparámetro o los datos no alcanzan para fijarlo.

In [ ]:
display(params.groupby(["model", "method", "params"], sort=False).size().rename("folds").to_frame().rename(index=name))

## 8. Comparación entre técnicas de segmentación

Esta es la pregunta del proyecto. Como los folds son los mismos para todas las técnicas, la comparación es **pareada**: en cada fold se resta la AUC de la técnica de referencia (la primera de `METHODS`) a la AUC de cada técnica, sobre exactamente las mismas imágenes de prueba.

Cómo leer la tabla, con el mismo criterio que la comparación entre arquitecturas de `02_segmentacion.ipynb`:
- **Diferencia media** frente a la **desviación de la diferencia**: si la diferencia media es menor que su desviación, la técnica no cambia el desempeño de forma consistente.
- **Folds en que gana**: una técnica que mejora en 5 de 5 folds es más convincente que una que mejora en 3 de 5 con la misma media.

Con 5 folds esta comparación es exploratoria. Para una conclusión firme hace falta repetir la validación cruzada con varias semillas de partición (sección 11).

In [ ]:
REFERENCE = METHODS[0]
if len(METHODS) < 2:
    print("Se necesita más de una técnica en METHODS para comparar.")
else:
    rows = []
    for model_name in MODELS:
        auc = metrics[metrics["model"] == model_name].pivot(index="fold", columns="method", values="auc")
        for method in METHODS[1:]:
            diff = auc[method] - auc[REFERENCE]
            rows.append({"modelo": name(model_name), "técnica": name(method),
                         f"AUC {name(REFERENCE)}": auc[REFERENCE].mean(), "AUC técnica": auc[method].mean(),
                         "diferencia media": diff.mean(), "desviación de la diferencia": diff.std(),
                         "folds en que gana": f"{(diff > 0).sum()} de {len(diff)}"})
    print(f"Referencia: {name(REFERENCE)}")
    display(pd.DataFrame(rows).set_index(["modelo", "técnica"]).round(4))

## 9. Importancia de características

Importancia de Random Forest (reducción media de impureza), promediada entre los 5 modelos de los folds externos. Responde una pregunta complementaria: **¿la técnica de segmentación cambia qué características usa el modelo?**, aunque la AUC no cambie.

Precaución: varias características de primer orden están muy correlacionadas (p. ej. `Mean`, `Median` y `RootMeanSquared`). Random Forest reparte la importancia entre las correlacionadas, así que una característica con importancia baja no es necesariamente irrelevante.

In [ ]:
TOP_K = 10
if importances.empty:
    print("Ningún modelo de MODELS expone feature_importances_.")
for model_name, imp in importances.groupby("model", sort=False):
    mean_imp = imp.groupby(["feature", "method"])["importance"].mean().unstack("method")[METHODS]
    top = mean_imp.mean(axis=1).nlargest(TOP_K).index
    data = mean_imp.loc[top]

    fig, ax = plt.subplots(figsize=(1.6 * len(METHODS) + 4, 0.38 * len(top) + 1.2))
    im = ax.imshow(data.to_numpy(), cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(METHODS)), [name(m) for m in METHODS])
    ax.set_yticks(range(len(top)), [f.split("_", 1)[1] for f in top])
    for (i, j), v in np.ndenumerate(data.to_numpy()):
        ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8,
                color="white" if v > data.to_numpy().max() * 0.6 else "black")
    fig.colorbar(im, ax=ax, label="importancia media")
    fig.suptitle(f"{name(model_name)}: {TOP_K} características más importantes", x=0.01, ha="left")
    fig.tight_layout()
    plt.show()

## 10. Guardado

Se guardan cuatro archivos en `results/classification/<dataset>/<tipos de imagen>_<familias>/`. La carpeta incluye la configuración de características para que dos experimentos (p. ej. solo primer orden frente a todas las familias) no se sobrescriban.

| Archivo | Contenido |
|---|---|
| `predictions.csv` | Puntaje y predicción de cada imagen, hechos por el modelo del fold en que la imagen fue prueba |
| `fold_metrics.csv` | Métricas de cada técnica × modelo × fold externo |
| `best_params.csv` | Hiperparámetros elegidos en cada fold externo |
| `importances.csv` | Importancia de cada característica por fold (modelos que la exponen) |

In [ ]:
CLF_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
outputs = {"predictions.csv": predictions, "fold_metrics.csv": metrics,
           "best_params.csv": params, "importances.csv": importances}
for file_name, df in outputs.items():
    df.to_csv(CLF_RESULTS_DIR / file_name, index=False)
print(f"Guardado en {CLF_RESULTS_DIR.relative_to(ROOT)}: {', '.join(outputs)}")

## 11. Qué hay que tener en cuenta

**Antes de ejecutar**
1. El notebook de extracción escribe un CSV por técnica y familia en `results/radiomics/<dataset>/` con el nombre y el formato de la sección 2.
2. Ajustar `METHODS` a las técnicas que tengan archivo.
3. Ejecutar todo. La celda de carga se detiene si falta un archivo, si falta una imagen o si un archivo no tiene columnas de su familia.

**Datos**
- **Máscaras de Montgomery-Shenzhen**: U-Net y Attention U-Net se entrenaron con el 70 % de estas imágenes (`segtrain_split.csv`), y MedSAM se desarrolló con este dataset (`tutorial_medsam.ipynb`, sección 11). En esas imágenes las máscaras serán mejores que en una radiografía nueva. No hay fuga de la etiqueta TB, porque las redes de segmentación no la usan para aprender, pero la calidad de la máscara no representa la del método en datos nuevos. Opciones: reportar también por partición de segmentación (`train` / `val` / `test`) o evaluar en TBX11K.
- **Etiqueta TB heterogénea en Montgomery-Shenzhen**: incluye casos activos, curados y en tratamiento (exploración, sección 8).
- **Folds de Montgomery-Shenzhen**: se calculan en este notebook y son reproducibles con la misma semilla y la misma versión de scikit-learn. Si otro notebook los necesita, conviene guardarlos en `data/metadata/`, como `subset.csv`.

**Protocolo**
- **5 folds son pocos para concluir.** La comparación de la sección 8 se vuelve firme repitiendo la validación cruzada con varias semillas de partición (como las tres semillas de la segmentación).
- **Umbral**: las métricas dependientes del umbral usan el umbral por defecto. Si el proyecto fija una sensibilidad objetivo (tamizaje), el umbral debe elegirse dentro del entrenamiento de cada fold, nunca en el fold de prueba.
- **Técnicas de referencia sugeridas**: `manual` (máscaras manuales, solo Montgomery-Shenzhen) y una sin segmentación (máscara de toda la imagen). Acotan el efecto: ¿cuánto se pierde frente a la máscara ideal y cuánto aporta segmentar?
- **Nuevos clasificadores**: agregar una entrada a `MODELS` (sección 5). Si el modelo necesita características escaladas, incluir `StandardScaler` en su pipeline.